In [43]:
# Import the pdfs
from pypdf import PdfReader
import pymupdf

# Vector Embedding
from qdrant_client import models, QdrantClient
from sentence_transformers import SentenceTransformer

# AI LLM
from dotenv import load_dotenv
load_dotenv()

True

In [ ]:


pdf_list = ['SRD_CC_v5.2.pdf', 'SRD-OGL_V1.1.pdf']
reader = []
for file in pdf_list:
    reader.append(PdfReader(file))
    print(f"Number of pages in {file}: {len(reader[-1].pages)}")
    
    

Ignoring wrong pointing object 8 0 (offset 0)
Ignoring wrong pointing object 10 0 (offset 0)
Ignoring wrong pointing object 13 0 (offset 0)
Ignoring wrong pointing object 25 0 (offset 0)
Ignoring wrong pointing object 35 0 (offset 0)
Ignoring wrong pointing object 41 0 (offset 0)
Ignoring wrong pointing object 51 0 (offset 0)
Ignoring wrong pointing object 236 0 (offset 0)


Number of pages in SRD_CC_v5.2.pdf: 361
Number of pages in SRD-OGL_V1.1.pdf: 398


In [13]:
reader[1].pages[2].extract_text()

'Not\t\r \xa0for\t\r \xa0resale.\t\r \xa0Permission\t\r \xa0granted\t\r \xa0to\t\r \xa0print\t\r \xa0or\t\r \xa0photocopy\t\r \xa0this\t\r \xa0document\t\r \xa0for\t\r \xa0personal\t\r \xa0use\t\r \xa0only.\t\r \xa0\t\r \xa0 System\t\r \xa0Reference\t\r \xa0Document\t\r \xa05.0\t\r \xa0  3\t\r \xa0\nIf\t\r \xa0you\t\r \xa0note\t\r \xa0any\t\r \xa0errors\t\r \xa0in\t\r \xa0this\t\r \xa0document,\t\r \xa0please\t\r \xa0let\t\r \xa0us\t\r \xa0know\t\r \xa0by\t\r \xa0\nemailing\t\r \xa0askdnd@wizards.com.\t\r \xa0\nRaces Racial Traits \nThe\t\r \xa0description\t\r \xa0of\t\r \xa0each\t\r \xa0race\t\r \xa0includes\t\r \xa0racial\t\r \xa0traits\t\r \xa0\nthat\t\r \xa0are\t\r \xa0common\t\r \xa0to\t\r \xa0members\t\r \xa0of\t\r \xa0that\t\r \xa0race.\t\r \xa0The\t\r \xa0\nfollowing\t\r \xa0entries\t\r \xa0appear\t\r \xa0among\t\r \xa0the\t\r \xa0traits\t\r \xa0of\t\r \xa0most\t\r \xa0\nraces.\t\r \xa0\nAbility Score Increase \nEvery\t\r \xa0race\t\r \xa0increases\t\r \xa0one\t\r \xa0or\t\r \x

In [3]:


pdf_list = ['SRD_CC_v5.2.pdf', 'SRD-OGL_V1.1.pdf']
reader = []
for file in pdf_list:
    document = pymupdf.open(file)
    full_text = []
    
    for page_num, page in enumerate(document):
        text = page.get_text("text")
        full_text.append(text)
    document.close()
    reader.append(full_text)
    

In [6]:
qdrant = QdrantClient(":memory:")
encoder = SentenceTransformer("all-MiniLM-L6-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

C:\Users\user\AppData\Roaming\Python\Python314\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\user\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
collection_name = "dnd_srd"

qdrant.recreate_collection(
    collection_name=collection_name,
    vectors_config=models.VectorParams(
        size=encoder.get_sentence_embedding_dimension(),
        distance=models.Distance.COSINE

    )
)

C:\Users\user\AppData\Local\Temp\ipykernel_32468\871876570.py:6: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  size=encoder.get_sentence_embedding_dimension(),
C:\Users\user\AppData\Local\Temp\ipykernel_32468\871876570.py:3: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  qdrant.recreate_collection(


True

In [ ]:
# Embeds the document into the vector space
qdrant.upload_points(
    collection_name=collection_name,
    points=[
        models.PointStruct(
            id=idx,
            vector=encoder.encode(text),
            payload={"text": text}
        ) for idx, text in enumerate(reader[1] + reader[0])
    ]
)

In [11]:
user_prompt = "What is a wizard?"

query_vector = encoder.encode(user_prompt)

In [16]:

hits = qdrant.query_points(
    collection_name=collection_name,
    query=query_vector,
    limit=5
)

In [37]:
for point in hits.points:
    print("ID:", point.id)
    print("Score:", point.score)
    print("Payload:", point.payload)
    print("-" * 50)

ID: 51
Score: 0.4423393010307149
Payload: {'text': 'Not\t\r \xa0for\t\r \xa0resale.\t\r \xa0Permission\t\r \xa0granted\t\r \xa0to\t\r \xa0print\t\r \xa0or\t\r \xa0photocopy\t\r \xa0this\t\r \xa0document\t\r \xa0for\t\r \xa0personal\t\r \xa0use\t\r \xa0only.\t\r \xa0\t\r \xa0\nSystem\t\r \xa0Reference\t\r \xa0Document\t\r \xa05.0\t\r \xa0\n 52\t\r \xa0\nWizard \nClass Features \nAs\t\r \xa0a\t\r \xa0wizard,\t\r \xa0you\t\r \xa0gain\t\r \xa0the\t\r \xa0following\t\r \xa0class\t\r \xa0features.\t\r \xa0\nHit Points \nHit\t\r \xa0Dice:\t\r \xa01d6\t\r \xa0per\t\r \xa0wizard\t\r \xa0level\t\r \xa0\nHit\t\r \xa0Points\t\r \xa0at\t\r \xa01st\t\r \xa0Level:\t\r \xa06\t\r \xa0+\t\r \xa0your\t\r \xa0Constitution\t\r \xa0\nmodifier\t\r \xa0\nHit\t\r \xa0Points\t\r \xa0at\t\r \xa0Higher\t\r \xa0Levels:\t\r \xa01d6\t\r \xa0(or\t\r \xa04)\t\r \xa0+\t\r \xa0your\t\r \xa0\nConstitution\t\r \xa0modifier\t\r \xa0per\t\r \xa0wizard\t\r \xa0level\t\r \xa0after\t\r \xa01st\t\r \xa0\nProficiencies \nArmor:\

In [48]:
import ollama

LANGUAGE_MODEL = "hf.co/bartowski/Llama-3.2-1B-Instruct-GGUF"

context_texts = [hit.payload["text"] for hit in hits.points]
context = "\n\n".join(context_texts)

In [49]:
prompt = f"""
You are a helpful assistant that answers questions about Dungeons and Dragons 5th Edition. Use the context provided to answer the question.

Context:
{context}

Question:
{user_prompt}

Answer:
"""

In [50]:


stream = ollama.chat(
    model=LANGUAGE_MODEL,
    messages=[
        {"role": "system", "content": prompt},
        {"role": "user", "content": user_prompt},
    ],
    stream=True,
)

print("\nChatbot response:")
for chunk in stream:
    print(chunk["message"]["content"], end="", flush=True)
print()


Chatbot response:
A wizard is a magical being in fiction, folklore, and popular culture. In the context of the text you provided, a wizard is a type of character who specializes in magic, particularly evocation, which involves creating powerful elemental effects such as fire, water, earth, and air.

In this context, a wizard is a member of a group of magical practitioners who have mastered the art of casting spells to affect the physical world. Wizards are often depicted as having a deep understanding of magic and its underlying principles, and they may use their powers to achieve various goals, such as protecting others, casting spells, or exploring the unknown.

Some common characteristics associated with wizards include:

* Mastery of magic, particularly evocation
* Expertise in spellcasting and incantations
* Ability to create and manipulate magical effects
* Often depicted as having a high level of intelligence and magical prowess
* May have a strong connection to nature and the 

In [51]:
def query_and_answer(user_prompt):
    query_vector = encoder.encode(user_prompt)
    hits = qdrant.query_points(
        collection_name=collection_name,
        query=query_vector,
        limit=5
    )
    
    context_texts = [hit.payload["text"] for hit in hits.points]
    context = "\n\n".join(context_texts)

    prompt = f"""
    You are a helpful assistant that answers questions about Dungeons and Dragons 5th Edition. Use the context provided to answer the question.

    Context:
    {context}

    Question:
    {user_prompt}

    Answer:
    """

    stream = ollama.chat(
        model=LANGUAGE_MODEL,
        messages=[
            {"role": "system", "content": prompt},
            {"role": "user", "content": user_prompt},
        ],
        stream=True,
    )

    print("\nChatbot response:")
    for chunk in stream:
        print(chunk["message"]["content"], end="", flush=True)
    print()

In [52]:
query_and_answer("What is a dwarf in Dungeons and Dragons?")


Chatbot response:
In Dungeons and Dragons (D&D), a dwarf is a playable character class. Dwarves are a subrace of humans, characterized by their physical strength, durability, and resilience.

Here are some key traits and characteristics of dwarves in D&D:

**Physical Characteristics:**

* Dwarves are sturdy and strong, with a strong bone structure.
* They have a robust physique, with a high percentage of hit points (HP) and a relatively low risk of injury.
* They often have a distinctive red beard or braided hair.

**Ability Scores:**

* Dwarves tend to have a high score in Strength (STR), which represents their physical power and muscularity.
* They also have a moderate score in Dexterity (DEX), which represents their agility and balance.

**Personality and Culture:**

* Dwarves are proud and traditional, with a strong sense of honor and loyalty.
* They are skilled craftsmen and traders, and often have a strong connection to the earth and its resources.
* They are fiercely independen